# 🧬 Notebook 1 : La régression, pas à pas

**Objectif** : comprendre les concepts fondamentaux du machine learning supervisé à travers un exemple de **régression** transposable en bio-informatique.

**Fil rouge biologique** : on veut prédire le **niveau d'expression d'un gène cible** à partir de la **concentration d'un facteur de transcription (TF)** qui le régule.

## Plan
1. Les données et la visualisation
2. Entraînement / validation / test
3. La descente de gradient (from scratch)
4. Régression linéaire avec scikit-learn
5. Les métriques d'erreur : MSE, RMSE, MAE, R²
6. Sous-apprentissage, sur-apprentissage, biais et variance
7. La courbe de validation
8. La courbe d'apprentissage
9. La validation croisée
10. Bonus : la régularisation (Ridge / Lasso)

## 0. Imports

In [ ]:
import numpy as np                 # calcul numérique
import pandas as pd                # tableaux de données
import matplotlib.pyplot as plt    # graphiques
import seaborn as sns              # graphiques statistiques

from sklearn.model_selection import train_test_split, cross_val_score, learning_curve, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")   # style des graphiques
rng = np.random.default_rng(42)    # générateur aléatoire reproductible (même résultat à chaque exécution)

## 1. Les données

On simule une relation **linéaire bruitée** : plus le facteur de transcription est concentré, plus le gène cible est exprimé.

- **X** (variable explicative / *feature*) : concentration du TF (unités arbitraires)
- **y** (variable cible / *target*) : expression du gène cible (par ex. log2 TPM)

> En vrai projet, ces données viendraient d'un fichier (RNA-seq, qPCR...). Ici on les simule pour **connaître la vérité** et mieux comprendre ce que fait le modèle.

In [ ]:
n = 200
tf = rng.uniform(0, 10, n)                       # concentration du facteur de transcription
bruit = rng.normal(0, 2.0, n)                    # bruit biologique + technique (écart-type = 2)
expression = 2.5 + 1.8 * tf + bruit              # vérité : y = 2.5 + 1.8 * x + bruit

df = pd.DataFrame({"tf": tf, "expression": expression})
print(df.head())
print(df.describe().round(2))

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(df["tf"], df["expression"], alpha=0.6)
plt.xlabel("Concentration du facteur de transcription (u.a.)")
plt.ylabel("Expression du gène cible (log2 TPM)")
plt.title("Relation TF → gène cible")
plt.show()

## 2. Entraînement / Validation / Test

On ne doit **jamais** évaluer un modèle sur les données qui ont servi à l'entraîner : il pourrait les avoir « apprises par cœur ».

| Jeu | Rôle | Analogie |
|---|---|---|
| **Entraînement (train)** | Le modèle apprend ses paramètres | Les cours |
| **Validation** | On compare des modèles / on règle les hyperparamètres | Les examens blancs |
| **Test** | Évaluation finale, **utilisé une seule fois** | L'examen final |

Ici : 60 % train, 20 % validation, 20 % test.

In [ ]:
X = df[["tf"]].values        # scikit-learn attend un tableau 2D pour X (n_échantillons, n_variables)
y = df["expression"].values  # et un tableau 1D pour y

# 1) on met de côté 20 % pour le test final
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# 2) on sépare le reste en train (60 % du total) et validation (20 % du total)
#    0.25 * 80 % = 20 %
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42)

print(f"Train : {len(X_train)} | Validation : {len(X_val)} | Test : {len(X_test)}")

## 3. La descente de gradient (from scratch)

**Idée** : un modèle linéaire s'écrit `ŷ = w·x + b`. On cherche `w` et `b` qui **minimisent l'erreur** (la *loss*, ici la MSE).

Imagine une vallée : la loss est l'altitude, `w` et `b` sont ta position. Le **gradient** indique la pente ; on **descend** à petits pas :

```
w ← w − η · ∂Loss/∂w
b ← b − η · ∂Loss/∂b
```

- **η (eta)** = *learning rate* (taille du pas). Trop grand → on diverge ; trop petit → c'est très lent.
- **Epoch** = un passage complet sur les données d'entraînement.

Pour la MSE = moyenne((ŷ − y)²) :
- ∂Loss/∂w = 2 · moyenne((ŷ − y) · x)
- ∂Loss/∂b = 2 · moyenne(ŷ − y)

In [ ]:
# On standardise x (moyenne 0, écart-type 1) : la descente de gradient converge bien mieux
mu, sigma = X_train.mean(), X_train.std()
x_tr = (X_train[:, 0] - mu) / sigma

def descente_gradient(x, y, lr=0.1, n_epochs=100):
    # Retourne w, b et l'historique de la loss
    w, b = 0.0, 0.0              # on part de paramètres arbitraires
    historique = []
    for epoch in range(n_epochs):
        y_pred = w * x + b                       # 1) prédiction
        erreur = y_pred - y                      # 2) erreur
        loss = np.mean(erreur ** 2)              # 3) MSE
        historique.append(loss)
        grad_w = 2 * np.mean(erreur * x)         # 4) gradients
        grad_b = 2 * np.mean(erreur)
        w -= lr * grad_w                         # 5) mise à jour (un pas de descente)
        b -= lr * grad_b
    return w, b, historique

w, b, hist = descente_gradient(x_tr, y_train, lr=0.1, n_epochs=100)
print(f"w = {w:.3f}, b = {b:.3f}, loss finale = {hist[-1]:.3f}")

In [ ]:
# Courbe de la loss : elle doit DESCENDRE puis se stabiliser (convergence)
plt.figure(figsize=(7, 4))
plt.plot(hist)
plt.xlabel("Epoch"); plt.ylabel("Loss (MSE)")
plt.title("Convergence de la descente de gradient")
plt.show()

In [ ]:
# Effet du learning rate : trop petit = lent, bon = rapide, trop grand = ça diverge
plt.figure(figsize=(7, 4))
for lr in [0.01, 0.1, 0.9, 1.05]:
    _, _, h = descente_gradient(x_tr, y_train, lr=lr, n_epochs=30)
    plt.plot(h, label=f"lr = {lr}")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss (échelle log)")
plt.legend(); plt.title("Influence du learning rate")
plt.show()

## 4. Régression linéaire avec scikit-learn

scikit-learn résout le même problème directement (de façon analytique). Le flux est toujours le même :

1. `modele = Modele()` : on crée le modèle
2. `modele.fit(X_train, y_train)` : on **entraîne**
3. `modele.predict(X)` : on **prédit**

In [ ]:
modele = LinearRegression()
modele.fit(X_train, y_train)

print(f"Pente (w)         : {modele.coef_[0]:.3f}   (vérité : 1.8)")
print(f"Ordonnée (b)      : {modele.intercept_:.3f}   (vérité : 2.5)")

# Vérification : notre descente de gradient donne la même droite (après dé-standardisation)
w_reel = w / sigma
b_reel = b - w * mu / sigma
print(f"Gradient descent  : w = {w_reel:.3f}, b = {b_reel:.3f}  → quasiment identique ✔")

In [ ]:
xs = np.linspace(0, 10, 100).reshape(-1, 1)
plt.figure(figsize=(7, 4.5))
plt.scatter(X_train, y_train, alpha=0.5, label="Train")
plt.scatter(X_val, y_val, alpha=0.7, color="orange", label="Validation")
plt.plot(xs, modele.predict(xs), color="red", lw=2, label="Modèle")
plt.xlabel("TF"); plt.ylabel("Expression"); plt.legend()
plt.title("Régression linéaire")
plt.show()

## 5. Les métriques d'erreur

Soit `yᵢ` la vraie valeur, `ŷᵢ` la prédiction, `n` le nombre d'échantillons :

| Métrique | Formule | Interprétation |
|---|---|---|
| **MSE** | moyenne((yᵢ − ŷᵢ)²) | Pénalise fortement les grosses erreurs. Unité = unité² |
| **RMSE** | √MSE | Même unité que y → **plus facile à interpréter** |
| **MAE** | moyenne(\|yᵢ − ŷᵢ\|) | Erreur moyenne absolue, robuste aux outliers |
| **R²** | 1 − SS_res / SS_tot | Part de variance expliquée (1 = parfait, 0 = pas mieux que la moyenne) |

**La loss** est la fonction que l'on minimise pendant l'entraînement (ici la MSE). Une **métrique** sert à évaluer : parfois c'est la même chose, parfois non.

In [ ]:
def evaluer(modele, X, y, nom=""):
    y_pred = modele.predict(X)
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    print(f"{nom:12s} MSE={mse:6.2f} | RMSE={rmse:5.2f} | MAE={mae:5.2f} | R²={r2:5.3f}")

evaluer(modele, X_train, y_train, "Train")
evaluer(modele, X_val, y_val, "Validation")
print("\nLe bruit injecté avait un écart-type de 2 → un RMSE proche de 2 est le mieux possible !")

In [ ]:
# Les résidus (erreurs) doivent être centrés sur 0, sans structure
residus = y_val - modele.predict(X_val)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(modele.predict(X_val), residus, alpha=0.7)
ax[0].axhline(0, color="red")
ax[0].set_xlabel("Valeurs prédites"); ax[0].set_ylabel("Résidus"); ax[0].set_title("Résidus vs prédictions")
sns.histplot(residus, kde=True, ax=ax[1])
ax[1].set_title("Distribution des résidus")
plt.show()

## 6. Sous-apprentissage, sur-apprentissage, biais et variance

- **Biais** : erreur due à un modèle **trop simple** qui rate la vraie structure → *sous-apprentissage (underfitting)*. Erreur élevée **même sur le train**.
- **Variance** : sensibilité du modèle aux **particularités** de l'échantillon d'entraînement → *sur-apprentissage (overfitting)*. Erreur très faible sur le train mais **élevée sur la validation**.

C'est le fameux **compromis biais-variance** : plus un modèle est complexe, plus le biais baisse, mais plus la variance monte.

Pour l'illustrer, on prend une relation **non linéaire** (courbe dose-réponse) avec **peu de données** (cas fréquent en biologie !).

In [ ]:
n2 = 60
dose = np.sort(rng.uniform(0, 10, n2))
reponse = 10 + 8 * np.sin(dose / 2.5) + rng.normal(0, 1.5, n2)   # vraie relation non linéaire

Xd = dose.reshape(-1, 1)
Xd_tr, Xd_val, yd_tr, yd_val = train_test_split(Xd, reponse, test_size=0.3, random_state=0)

def modele_poly(degre):
    # Régression polynomiale : on ajoute x², x³, ... comme nouvelles variables
    return make_pipeline(PolynomialFeatures(degre, include_bias=False), StandardScaler(), LinearRegression())

xs = np.linspace(0, 10, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, degre, titre in zip(axes, [1, 3, 15], ["Sous-apprentissage (degré 1)", "Bon compromis (degré 3)", "Sur-apprentissage (degré 15)"]):
    m = modele_poly(degre).fit(Xd_tr, yd_tr)
    ax.scatter(Xd_tr, yd_tr, label="Train")
    ax.scatter(Xd_val, yd_val, color="orange", label="Validation")
    ax.plot(xs, m.predict(xs), color="red", lw=2)
    rmse_tr = np.sqrt(mean_squared_error(yd_tr, m.predict(Xd_tr)))
    rmse_val = np.sqrt(mean_squared_error(yd_val, m.predict(Xd_val)))
    ax.set_title(f"{titre}\nRMSE train={rmse_tr:.2f} | val={rmse_val:.2f}")
    ax.set_ylim(-5, 25)
axes[0].legend()
plt.show()

## 7. La courbe de validation

On fait varier **un hyperparamètre** (ici le degré du polynôme = la complexité) et on regarde l'erreur train vs validation.

- Zone de gauche : les deux erreurs sont élevées → **biais** (underfitting)
- Zone de droite : train bas, validation qui remonte → **variance** (overfitting)
- Le **minimum de l'erreur de validation** indique la bonne complexité.

In [ ]:
degres = range(1, 16)
err_tr, err_val = [], []
for d in degres:
    m = modele_poly(d).fit(Xd_tr, yd_tr)
    err_tr.append(np.sqrt(mean_squared_error(yd_tr, m.predict(Xd_tr))))
    err_val.append(np.sqrt(mean_squared_error(yd_val, m.predict(Xd_val))))

plt.figure(figsize=(8, 4.5))
plt.plot(degres, err_tr, "o-", label="Train")
plt.plot(degres, err_val, "o-", label="Validation")
meilleur = list(degres)[int(np.argmin(err_val))]
plt.axvline(meilleur, color="gray", ls="--", label=f"Meilleur degré = {meilleur}")
plt.xlabel("Complexité du modèle (degré du polynôme)")
plt.ylabel("RMSE")
plt.title("Courbe de validation")
plt.legend(); plt.ylim(0, 8)
plt.show()

## 8. La courbe d'apprentissage

On fait varier cette fois **la quantité de données d'entraînement** et on regarde les erreurs.

**Comment la lire ?**
- Les deux courbes sont **hautes et proches** → biais élevé : plus de données n'aidera pas, il faut un **modèle plus riche**.
- **Grand écart** entre train (bas) et validation (haute) → variance élevée : **plus de données** ou un modèle **plus simple / régularisé** aideront.

In [ ]:
def tracer_courbe_apprentissage(modele, X, y, titre):
    tailles, scores_tr, scores_val = learning_curve(
        modele, X, y,
        train_sizes=np.linspace(0.15, 1.0, 8),
        cv=5,
        scoring="neg_mean_squared_error",   # scikit-learn maximise un score → on prend -MSE
    )
    rmse_tr = np.sqrt(-scores_tr.mean(axis=1))
    rmse_val = np.sqrt(-scores_val.mean(axis=1))
    plt.plot(tailles, rmse_tr, "o-", label="Train")
    plt.plot(tailles, rmse_val, "o-", label="Validation (CV)")
    plt.xlabel("Nombre d'échantillons d'entraînement"); plt.ylabel("RMSE")
    plt.title(titre); plt.legend(); plt.ylim(0, 8)

plt.figure(figsize=(14, 4.5))
plt.subplot(1, 3, 1); tracer_courbe_apprentissage(modele_poly(1), Xd, reponse, "Degré 1 : biais élevé")
plt.subplot(1, 3, 2); tracer_courbe_apprentissage(modele_poly(3), Xd, reponse, "Degré 3 : bon modèle")
plt.subplot(1, 3, 3); tracer_courbe_apprentissage(modele_poly(12), Xd, reponse, "Degré 12 : variance élevée")
plt.tight_layout(); plt.show()

## 9. La validation croisée (cross-validation)

Problème : avec peu de données, un seul découpage train/validation est **chanceux ou malchanceux**. Le résultat dépend du hasard du tirage.

**K-fold CV** : on coupe les données en *K* blocs (*folds*). On entraîne K fois, chaque bloc servant tour à tour de validation. On **moyenne** les K scores → estimation plus **stable et fiable**.

```
Fold 1 : [VAL][ train ][ train ][ train ][ train ]
Fold 2 : [train][ VAL ][ train ][ train ][ train ]
...
```

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)   # 5 blocs, mélangés

for d in [1, 3, 12]:
    # scoring = -MSE → on repasse en RMSE positif
    scores = cross_val_score(modele_poly(d), Xd, reponse, cv=kf, scoring="neg_mean_squared_error")
    rmse = np.sqrt(-scores)
    print(f"Degré {d:2d} : RMSE par fold = {np.round(rmse, 2)} | moyenne = {rmse.mean():.2f} ± {rmse.std():.2f}")

## 10. Bonus : la régularisation (Ridge / Lasso)

Pour limiter le sur-apprentissage sans changer le degré, on **pénalise les coefficients trop grands** :

- **Ridge (L2)** : ajoute `α · Σ w²` à la loss → coefficients **petits** mais non nuls.
- **Lasso (L1)** : ajoute `α · Σ |w|` à la loss → certains coefficients **tombent à 0** = **sélection de variables**.

En génomique (20 000 gènes pour 100 patients), le Lasso est très utilisé pour sélectionner les gènes pertinents.

In [ ]:
# Degré 12 (qui sur-apprend) avec et sans régularisation
modeles = {
    "Sans régularisation": make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), LinearRegression()),
    "Ridge (α=1)":         make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), Ridge(alpha=1.0)),
    "Lasso (α=0.1)":       make_pipeline(PolynomialFeatures(12, include_bias=False), StandardScaler(), Lasso(alpha=0.1, max_iter=50000)),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharey=True)
for ax, (nom, m) in zip(axes, modeles.items()):
    m.fit(Xd_tr, yd_tr)
    rmse_val = np.sqrt(mean_squared_error(yd_val, m.predict(Xd_val)))
    ax.scatter(Xd_tr, yd_tr); ax.scatter(Xd_val, yd_val, color="orange")
    ax.plot(xs, m.predict(xs), color="red", lw=2)
    ax.set_title(f"{nom}\nRMSE validation = {rmse_val:.2f}"); ax.set_ylim(-5, 25)
plt.show()

## 11. Évaluation finale sur le jeu de TEST

Une fois le modèle choisi (grâce à la validation / CV), on l'évalue **une seule fois** sur le jeu de test mis de côté au début. On ré-entraîne souvent sur train + validation avant.

In [ ]:
# Sur notre premier jeu de données (TF → gène cible)
modele_final = LinearRegression().fit(np.vstack([X_train, X_val]), np.concatenate([y_train, y_val]))
evaluer(modele_final, X_test, y_test, "TEST final")

## ✅ Récapitulatif

| Concept | À retenir |
|---|---|
| Train / Val / Test | Apprendre / choisir / évaluer honnêtement |
| Loss | Ce que l'on minimise pendant l'entraînement |
| Descente de gradient | On descend la pente de la loss, pas à pas (learning rate η) |
| MSE / RMSE / MAE / R² | Mesurer l'erreur de prédiction |
| Biais | Modèle trop simple → underfitting |
| Variance | Modèle trop complexe → overfitting |
| Courbe de validation | Erreur vs complexité du modèle |
| Courbe d'apprentissage | Erreur vs taille des données |
| Validation croisée | Estimation fiable avec peu de données |
| Ridge / Lasso | Régularisation, sélection de variables |

## 🧪 Exercices
1. Change `lr` dans la descente de gradient (essaie 0.001 et 1.5) : que se passe-t-il ?
2. Augmente le bruit (`rng.normal(0, 5, n)`) : comment évoluent le RMSE et le R² ?
3. Ajoute une 2ᵉ variable explicative (ex. méthylation du promoteur) et utilise `LinearRegression` sur 2 colonnes.
4. Fais varier `alpha` pour Ridge (0.001 → 100) et trace la courbe de validation correspondante.